# Stimulus Spending Parser — ARRA & CARES Act

Reads directly from CBO PDFs — no hardcoded values. All figures are extracted from
the actual tables in the PDFs. Assertions verify extracted numbers against CBO's
own published headline totals before writing output.

**Setup — download these two PDFs into this directory:**
- `arra_cost_estimate.pdf` → https://www.cbo.gov/publication/41762
- `hr748.pdf` → https://www.cbo.gov/system/files/2020-04/hr748.pdf

**Output schema:** `law, fiscal_year, category, amount_billions, pct_gdp`

## Cell 1 — Imports and helpers

In [1]:
import pdfplumber
import pandas as pd
import openpyxl
import re

ARRA_PDF   = 'arra_cost_estimate.pdf'
CARES_PDF  = 'hr748.pdf'
OMB_FILE   = 'BUDGET-2026-HIST.xlsx'
OUTPUT_CSV = '../public/stimulus_spending.csv'

def to_num(s):
    """CBO cell -> float. '*' = 0, parentheses = negative, None = unparseable."""
    s = str(s or '').strip().replace(',', '')
    if s in ('', '*', '**', 'n.a.', '-'): return 0.0
    if s.startswith('(') and s.endswith(')'): return -float(s[1:-1])
    try: return float(s)
    except: return None

def find_year_cols(table, year_range):
    """Scan table rows for year integers; return {col_index: year} from first matching row."""
    for row in table:
        col_map = {}
        for j, cell in enumerate(row):
            try:
                yr = int(str(cell or '').strip())
                if year_range[0] <= yr <= year_range[1]:
                    col_map[j] = yr
            except:
                pass
        if col_map:
            return col_map
    return {}

def row_vals(row, col_map, years):
    """Extract {year: float} from a table row using col_map, filtering to requested years."""
    return {yr: to_num(row[j])
            for j, yr in col_map.items()
            if yr in years and to_num(row[j]) is not None}

print('Ready.')

Ready.


## Cell 2 — Inspect ARRA PDF

Run this to see which pages hold the summary cost table and what the row labels look like.

In [2]:
with pdfplumber.open(ARRA_PDF) as pdf:
    print(f'ARRA: {len(pdf.pages)} pages')
    for i, page in enumerate(pdf.pages[:8]):
        text = page.extract_text() or ''
        if any(k in text for k in ['Division A', 'Division B', 'Total Changes', 'Revenues']):
            print(f'\n--- Page {i+1} ---')
            print(text[:600])
            for t in page.extract_tables():
                col_map = find_year_cols(t, (2009, 2019))
                if not col_map:
                    continue
                print(f'  Table: {len(t)} rows, year cols: {col_map}')
                for row in t:
                    label = str(row[0] or '').strip()
                    if label:
                        print(f'    [{label[:60]}]', [row[j] for j in col_map])

ARRA: 8 pages

--- Page 3 ---
TABLE 1. SUMMARY OF ESTIMATED COST OF THE CONFERENCE AGREEMENT FOR H.R. 1, THE
AMERICAN RECOVERY AND REINVESTMENT ACT OF 2009, AS POSTED ON THE WEB SITE OF
THE HOUSE COMMITTEE ON RULES
By Fiscal Year, in Billions of Dollars
2009-
2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2019
DIVISION A—APPROPRIATIONS a
Estimated Budget Authority 288.7 7.1 4.6 3.6 2.5 1.1 1.1 1.1 1.1 0.5 0 311.2
Estimated Outlays 34.8 110.7 76.3 38.1 22.9 12.8 7.0 3.1 1.6 0.8 0.1 308.3
DIVISION A—REVENUES
Estimated Revenues * * * * * * * * * * * -0.1
DIVISION B—DIRECT SPENDING
Estimated Budget Authority 90.3 107.6

--- Page 4 ---
Table 2. ESTIMATED COST OF THE CONFERENCE AGREEMENT FOR H.R. 1, THE AMERICAN RECOVERY AND REINVESTMENT ACT OF 2009,
AS POSTED ON THE WEBSITE OF THE HOUSE COMMITTEE ON RULES
By Fiscal Year, Millions of Dollars
Total
2009 -
2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2019
Discretionary Spending 1/
Division A
Title I - Agriculture, Rural
Devel

## Cell 3 — Parse ARRA

Source: CBO pub. 41762 — H.R.1 conference agreement, Feb 13 2009  
https://www.cbo.gov/publication/41762

Looks for:
- **Division A outlays** — discretionary appropriations (infrastructure, grants, agencies)
- **Division B outlays** — mandatory/direct spending (UI, Medicaid, nutrition, refundable credits)
- **Total revenue reductions** — tax cuts (stored as positive)

Validation: 2009+2010 combined total should be near CBO headline of $185B + $399B = $584B.

In [3]:
ARRA_YEARS  = [2009, 2010, 2011, 2012, 2013]

discr_by_yr = {}  # Division A outlays (discretionary appropriations)
mand_by_yr  = {}  # Division B outlays (mandatory/direct spending)
rev_by_yr   = {}  # revenue reductions (stored as positive)

with pdfplumber.open(ARRA_PDF) as pdf:
    for i, page in enumerate(pdf.pages[:8]):
        for t in page.extract_tables():
            col_map = find_year_cols(t, (2009, 2019))
            if not col_map:
                continue
            for row in t:
                label = str(row[0] or '').strip().lower()

                if not discr_by_yr and 'division a' in label and 'outlay' in label:
                    vals = row_vals(row, col_map, ARRA_YEARS)
                    if vals:
                        discr_by_yr = vals
                        print(f'Page {i+1} Division A (discretionary) outlays: {discr_by_yr}')

                if not mand_by_yr and 'division b' in label and 'outlay' in label:
                    vals = row_vals(row, col_map, ARRA_YEARS)
                    if vals:
                        mand_by_yr = vals
                        print(f'Page {i+1} Division B (mandatory) outlays: {mand_by_yr}')

                if not rev_by_yr and 'total' in label and 'revenue' in label:
                    vals = {yr: abs(v)
                            for yr, v in row_vals(row, col_map, ARRA_YEARS).items()}
                    if vals:
                        rev_by_yr = vals
                        print(f'Page {i+1} Revenue reductions: {rev_by_yr}')

# Assertions
assert discr_by_yr, 'FAIL: Could not parse Division A outlays — review Cell 2 output'
assert mand_by_yr,  'FAIL: Could not parse Division B outlays — review Cell 2 output'
assert rev_by_yr,   'FAIL: Could not parse revenue reductions — review Cell 2 output'

combined_09_10 = sum(
    discr_by_yr.get(y, 0) + mand_by_yr.get(y, 0) + rev_by_yr.get(y, 0)
    for y in [2009, 2010]
)
print(f'\n2009+2010 combined: ${combined_09_10:.0f}B  (CBO headline: ~$584B)')
assert 450 < combined_09_10 < 750, \
    f'FAIL: ${combined_09_10:.0f}B outside expected $450-$750B range'
print('All assertions passed.')

# Build DataFrame
arra_rows = []
for yr in ARRA_YEARS:
    arra_rows.append({'law': 'ARRA', 'fiscal_year': yr,
                      'category': 'Discretionary Outlays',
                      'amount_billions': discr_by_yr.get(yr, 0.0)})
    arra_rows.append({'law': 'ARRA', 'fiscal_year': yr,
                      'category': 'Mandatory Outlays',
                      'amount_billions': mand_by_yr.get(yr, 0.0)})
    arra_rows.append({'law': 'ARRA', 'fiscal_year': yr,
                      'category': 'Revenue Reductions',
                      'amount_billions': rev_by_yr.get(yr, 0.0)})

df_arra = pd.DataFrame(arra_rows)
totals  = df_arra.groupby(['law','fiscal_year'])['amount_billions'].sum().reset_index()
totals['category'] = 'Total Direct Cost'
df_arra = pd.concat([df_arra, totals], ignore_index=True)
print('\nARRA totals by year:')
print(df_arra[df_arra.category == 'Total Direct Cost'].to_string(index=False))

AssertionError: FAIL: Could not parse Division A outlays — review Cell 2 output

## Cell 4 — Inspect CARES Act PDF

Run this to locate Table 2 (direct spending), Table 3 (revenues), Table 4 (discretionary).  
These are on the last few pages of hr748.pdf.

In [ ]:
with pdfplumber.open(CARES_PDF) as pdf:
    print(f'CARES: {len(pdf.pages)} pages')
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        if any(k in text for k in ['Table 2', 'Table 3', 'Table 4',
                                    'Total Changes in Direct Spending',
                                    'Total Changes in Revenues']):
            print(f'\n--- Page {i+1} ---')
            print(text[:400])
            for t in page.extract_tables():
                col_map = find_year_cols(t, (2020, 2030))
                if not col_map:
                    continue
                print(f'  Table: {len(t)} rows, year cols: {col_map}')
                for row in t:
                    label = str(row[0] or '').strip()
                    if label:
                        print(f'    [{label[:70]}]', [row[j] for j in col_map])

## Cell 5 — Parse CARES Act

Source: CBO pub. 56334 — revised April 27 2020  
https://www.cbo.gov/publication/56334

Reads three tables from the PDF:
- **Table 2** total `Estimated Outlays` row → mandatory/direct spending by year
- **Table 3** `Total Changes in Revenues` row → revenue reductions by year (stored as positive)
- **Table 4** total outlays row → discretionary (Division B) by year

Validation checks against CBO p.2 headlines:
- Mandatory (Table 2): ~$988B over 2020-2030
- Revenue (Table 3): ~$408B over 2020-2030
- Discretionary (Table 4): ~$326B total

In [ ]:
CARES_YEARS   = [2020, 2021]

outlays_by_yr = {}  # Table 2: mandatory/direct spending
revenue_by_yr = {}  # Table 3: revenue reductions (positive)
discr_by_yr   = {}  # Table 4: discretionary (Division B)

found_t2 = False
found_t3 = False
found_t4 = False

with pdfplumber.open(CARES_PDF) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        for t in page.extract_tables():
            if not t or len(t) < 2:
                continue
            col_map = find_year_cols(t, (2020, 2030))
            if not col_map:
                continue

            for row in t:
                label = str(row[0] or '').strip().lower()

                # Table 2 grand total: 'Estimated Outlays' (not On-Budget / Off-Budget sub-rows)
                if (not found_t2
                        and 'estimated outlays' in label
                        and 'on-budget' not in label
                        and 'off-budget' not in label):
                    vals = row_vals(row, col_map, CARES_YEARS)
                    if vals and max(vals.values(), default=0) > 100:
                        outlays_by_yr = vals
                        found_t2 = True
                        print(f'Page {i+1} Table 2 mandatory outlays: {outlays_by_yr}')

                # Table 3: 'Total Changes in Revenues'
                if not found_t3 and 'total changes in revenues' in label:
                    vals = {yr: abs(v)
                            for yr, v in row_vals(row, col_map, CARES_YEARS).items()}
                    if vals:
                        revenue_by_yr = vals
                        found_t3 = True
                        print(f'Page {i+1} Table 3 revenue reductions: {revenue_by_yr}')

                # Table 4: total outlays row for discretionary Division B
                # Only look after Tables 2 and 3 are found, to avoid false matches
                if (not found_t4
                        and found_t2 and found_t3
                        and 'total' in label
                        and 'estimated outlays' not in label
                        and 'revenues' not in label):
                    vals = row_vals(row, col_map, CARES_YEARS)
                    if vals and 50 < max(vals.values(), default=0) < 600:
                        discr_by_yr = vals
                        found_t4 = True
                        print(f'Page {i+1} Table 4 discretionary outlays: {discr_by_yr}')

# Assertions
assert outlays_by_yr, 'FAIL: Could not parse Table 2 mandatory outlays — review Cell 4'
assert revenue_by_yr, 'FAIL: Could not parse Table 3 revenue reductions — review Cell 4'
assert discr_by_yr,   'FAIL: Could not parse Table 4 discretionary outlays — review Cell 4'

disc_total = sum(discr_by_yr.values())
print(f'\nValidation (CBO pub. 56334, p.2 — all are 2020-2030 totals):')
print(f'  Mandatory  FY2020=${outlays_by_yr.get(2020,0):.0f}B  FY2021=${outlays_by_yr.get(2021,0):.0f}B  (10yr ~$988B)')
print(f'  Revenue    FY2020=${revenue_by_yr.get(2020,0):.0f}B  FY2021=${revenue_by_yr.get(2021,0):.0f}B  (10yr ~$408B)')
print(f'  Discr      FY2020=${discr_by_yr.get(2020,0):.0f}B  FY2021=${discr_by_yr.get(2021,0):.0f}B  (total ~$326B)')

assert 800 < outlays_by_yr.get(2020, 0) < 1100, \
    f'FAIL: FY2020 mandatory ${outlays_by_yr.get(2020,0):.0f}B outside $800-$1100B'
assert 300 < revenue_by_yr.get(2020, 0) < 700, \
    f'FAIL: FY2020 revenue ${revenue_by_yr.get(2020,0):.0f}B outside $300-$700B'
assert 150 < disc_total < 450, \
    f'FAIL: Discretionary total ${disc_total:.0f}B outside $150-$450B'
print('All assertions passed.')

# Build DataFrame
cares_rows = []
for yr in CARES_YEARS:
    cares_rows.append({'law': 'CARES Act', 'fiscal_year': yr,
                       'category': 'Mandatory Outlays',
                       'amount_billions': outlays_by_yr.get(yr, 0.0)})
    cares_rows.append({'law': 'CARES Act', 'fiscal_year': yr,
                       'category': 'Discretionary Outlays',
                       'amount_billions': discr_by_yr.get(yr, 0.0)})
    cares_rows.append({'law': 'CARES Act', 'fiscal_year': yr,
                       'category': 'Revenue Reductions',
                       'amount_billions': revenue_by_yr.get(yr, 0.0)})

df_cares = pd.DataFrame(cares_rows)
totals   = df_cares.groupby(['law','fiscal_year'])['amount_billions'].sum().reset_index()
totals['category'] = 'Total Direct Cost'
df_cares = pd.concat([df_cares, totals], ignore_index=True)
print('\nCARES totals by year:')
print(df_cares[df_cares.category == 'Total Direct Cost'].to_string(index=False))

## Cell 6 — Load GDP from OMB Historical Tables (hist10z1)

Parses GDP (in millions) from BUDGET-2026-HIST.xlsx. Converted to billions for pct_gdp.

In [ ]:
wb       = openpyxl.load_workbook(OMB_FILE, read_only=True, data_only=True)
ws       = wb['hist10z1']
omg_rows = list(ws.iter_rows(values_only=True))

year_indices = []
years_omg    = []
header_idx   = None

for i, row in enumerate(omg_rows):
    cleaned = [str(c).strip("'") for c in row]
    if '2009' in cleaned and '2020' in cleaned:
        for j, c in enumerate(row):
            s = str(c).strip("'")
            if re.match(r'^\d{4}$', s) and 1929 <= int(s) <= 2030:
                years_omg.append(int(s))
                year_indices.append(j)
        header_idx = i
        print(f'OMB header at row {i}: {years_omg[:5]}...{years_omg[-3:]}')
        break

assert header_idx is not None, 'FAIL: Could not find year header in OMB hist10z1'

gdp_lookup = {}
for row in omg_rows[header_idx+1:]:
    label = str(row[0] or '').strip("'").lower()
    if 'gross domestic product' in label:
        for yr, idx in zip(years_omg, year_indices):
            val = row[idx]
            if val is not None:
                try: gdp_lookup[yr] = float(str(val).strip("'").replace(',',''))
                except: pass
        print('GDP row found.')
        break

assert gdp_lookup, 'FAIL: Could not find GDP row in OMB hist10z1'
for y in [2009, 2010, 2011, 2020, 2021]:
    g = gdp_lookup.get(y)
    print(f'  GDP {y}: ${g/1e6:.2f}T' if g else f'  WARN: GDP {y} not found')

## Cell 7 — Combine, compute % of GDP, final checks, write output

In [ ]:
df = pd.concat([df_arra, df_cares], ignore_index=True)

# GDP in millions -> billions
df['gdp_b']   = df['fiscal_year'].map(lambda y: gdp_lookup.get(y)) / 1000
df['pct_gdp'] = (df['amount_billions'] / df['gdp_b'] * 100).round(2)
df = df.drop(columns=['gdp_b'])
df = df.sort_values(['law', 'fiscal_year', 'category']).reset_index(drop=True)

print('Final totals by law and year:')
print(df[df.category == 'Total Direct Cost']
      [['law','fiscal_year','amount_billions','pct_gdp']].to_string(index=False))

# Final sanity checks
def total(law, yr):
    return df[(df.law==law) & (df.fiscal_year==yr) &
              (df.category=='Total Direct Cost')]['amount_billions'].values[0]

assert 100 < total('ARRA', 2009) < 300,    f'FAIL: ARRA 2009 = ${total("ARRA",2009):.0f}B'
assert 300 < total('ARRA', 2010) < 500,    f'FAIL: ARRA 2010 = ${total("ARRA",2010):.0f}B'
assert 1000 < total('CARES Act', 2020) < 2500, f'FAIL: CARES 2020 = ${total("CARES Act",2020):.0f}B'
print('\nAll final assertions passed.')

df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {OUTPUT_CSV}  ({len(df)} rows)')